In [9]:
import sqlite3
import pandas as pd

In [10]:
connection = sqlite3.connect("outputs/olist.db")

In [11]:
cursor = connection.cursor()

In [12]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
sonuc = cursor.fetchall()
sonuc

[]

In [13]:
orders_df = pd.read_csv("data/olist_orders_dataset.csv")
order_items_df = pd.read_csv("data/olist_order_items_dataset.csv")
customers_df = pd.read_csv("data/olist_customers_dataset.csv")
products_df = pd.read_csv("data/olist_products_dataset.csv")
payments_df = pd.read_csv("data/olist_order_payments_dataset.csv")
reviews_df = pd.read_csv("data/olist_order_reviews_dataset.csv")

In [22]:
tum_dfler = {
    "orders": orders_df,
    "order_items": order_items_df,
    "customers": customers_df,
    "products": products_df,
    "payments": payments_df,
    "reviews": reviews_df
}

In [24]:
for isim, df in tum_dfler.items():
    print(f"{isim}: {df.shape}")

orders: (99441, 8)
order_items: (112650, 7)
customers: (99441, 5)
products: (32951, 9)
payments: (103886, 5)
reviews: (99224, 7)


In [25]:
for isim, df in tum_dfler.items():
    df.to_sql(isim, connection, if_exists="replace", index=False)

In [26]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()

[('orders',),
 ('order_items',),
 ('customers',),
 ('products',),
 ('payments',),
 ('reviews',)]

In [27]:
cursor.execute("PRAGMA table_info(payments);")
cursor.fetchall()

[(0, 'order_id', 'TEXT', 0, None, 0),
 (1, 'payment_sequential', 'INTEGER', 0, None, 0),
 (2, 'payment_type', 'TEXT', 0, None, 0),
 (3, 'payment_installments', 'INTEGER', 0, None, 0),
 (4, 'payment_value', 'REAL', 0, None, 0)]

* **payments** tablosundan, ödeme değeri (`payment_value`) **500'den büyük** olan kayıtları, değeri **büyükten küçüğe** sıralı şekilde **ilk 10** tanesini getir.

In [28]:
cursor.execute("""
    SELECT order_id, payment_type, payment_value
    FROM payments
    WHERE payment_value > 500
    ORDER BY payment_value DESC
    LIMIT 10;
""")
sonuc = cursor.fetchall()
sonuc

[('03caa2c082116e1d31e67e9ae3700499', 'credit_card', 13664.08),
 ('736e1922ae60d0d6a89247b851902527', 'boleto', 7274.88),
 ('0812eb902a67711a1cb742b3cdaa65ae', 'credit_card', 6929.31),
 ('fefacc66af859508bf1a7934eab1e97f', 'boleto', 6922.21),
 ('f5136e38d1a14a4dbd87dff67da82701', 'boleto', 6726.66),
 ('2cc9089445046817a7539d90805e6e5a', 'boleto', 6081.54),
 ('a96610ab360d42a2e5335a3998b4718a', 'credit_card', 4950.34),
 ('b4c4b76c642808cbe472a32b86cddc95', 'credit_card', 4809.44),
 ('199af31afc78c699f0dbf71fb178d4d4', 'credit_card', 4764.34),
 ('8dbc85d1447242f3b127dda390d56e19', 'credit_card', 4681.78)]

* **payments** tablosunda, her `payment_type` için: kaç tane ödeme yapıldığı (`COUNT`) ve toplam ödeme değeri (`SUM`) nedir?

In [31]:
cursor.execute("""
    SELECT payment_type, COUNT(*) AS islem_sayisi, SUM(payment_value) AS toplam_deger
    FROM payments
    GROUP BY payment_type
    ORDER BY toplam_deger DESC;
""")
sonuc = cursor.fetchall()
sonuc

[('credit_card', 76795, 12542084.189999327),
 ('boleto', 19784, 2869361.2699999753),
 ('voucher', 5775, 379436.8700000007),
 ('debit_card', 1529, 217989.7900000001),
 ('not_defined', 3, 0.0)]

### Ödeme Yöntemi Dağılımı — Yorum

**İş sorusu:** Hangi ödeme yöntemleri en yüksek işlem sayısına ve toplam gelire sahip?

- **Credit card** açık ara en baskın ödeme yöntemi: 76.795 işlem, ~12.5M toplam değer — işlem sayısının %74'ü, toplam gelirin ~%78'ini oluşturuyor.
- **Boleto** (Brezilya'ya özgü bir tür banka havalesi/fatura ödeme yöntemi) ikinci sırada, ancak credit card'ın çok gerisinde kalıyor.
- `not_defined` olarak etiketlenmiş 3 satır var ve bunların değeri 0.0 — bu muhtemelen bir veri kalitesi sorununu (eksik/bozuk kayıt) işaret ediyor. Satır sayısı çok az olduğu için genel analizi etkilemiyor, ama sonuç/özet bölümünde veri setindeki bu tür küçük tutarsızlıklara değinmek gerekiyor.